# DATASCI 151 - Introduction to Statistical Computing II
## Lecture 13 - Merge Tables in SQL
**Author:** Danilo Freire (danilo.freire@emory.edu)
*Department of Data and Decision Sciences - Emory University*

# Hello again! 😊

# Brief recap 📚

## Recap of last class and today's plan

### Last time we learned how to:

- Connect SQL with Python with `sqlite3` and `pandas`
- Use many SQL commands, such as `CASE WHEN`, window functions, and string functions
- Fill missing data with `COALESCE` and `CASE WHEN`
- Use `pandas` to write and run SQL queries

### Today we will learn how to:

- See different types of join in SQL
- Use special joins, such as `CROSS JOIN` and `SELF JOIN`
- Merge tables by row with `UNION`, `INTERSECT`, and `EXCEPT`
- Learn how to create `UPSERT` statements in SQLite
- Create `VIEWS` in SQLite
- Solve exercises to practice what we learned
- Let's get started! 🚀

# Basic joins 📊

## Primary and foreign keys

- In SQL you can merge two tables either **by columns or by rows**
- The most common approach is the **`JOIN`** clause, which **combines rows and columns from two tables** on a related column
- Two types of keys, **primary and foreign**:
  - The **primary key** **uniquely identifies each row** in a table
  - A **foreign key** **refers to a primary key in another table**; a table can have several, and they can be `NULL`
- Note: SQLite does not enforce foreign-key constraints by default; run `PRAGMA foreign_keys = ON;` if you need enforcement

[![](figures/common_database_keys_explained-f_mobile.png){width="70%"}](#){data-modal-type="image" data-modal-url="figures/common_database_keys_explained-f_mobile.png"}

## Load the libraries and connect to the database

- Let's load the libraries and connect to the SQLite database. We'll use a file named `lecture13.db`

In [ ]:
import pandas as pd
import sqlite3

# Connect to the SQLite database (this will create the file if it doesn't exist)
connection = sqlite3.connect('lecture13.db')
cursor = connection.cursor()

cursor.execute('DROP TABLE IF EXISTS players;')
cursor.execute('''
CREATE TABLE players (
    player_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    player_name TEXT    NOT NULL UNIQUE,
    goals       INTEGER NOT NULL,
    victories   INTEGER NOT NULL
);
''')

cursor.execute('DROP TABLE IF EXISTS teams;')
cursor.execute('''
CREATE TABLE teams (
    team_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    team_name TEXT    NOT NULL
);
''')
connection.commit()

## Create the tables

- Now let's insert some data into the tables!

In [ ]:
# Insert data into the tables
cursor.execute('''
INSERT INTO players (player_name, goals, victories) VALUES
('Messi', 10, 5),
('Vini Jr', 8, 4),
('Neymar', 6, 3),
('Mbappe', 5, 2),
('Lewandowski', 4, 1),
('Haaland', 5, 3);
''')

cursor.execute('''
INSERT INTO teams (team_name) VALUES
('Inter Miami'),
('Real Madrid'),
('Santos'),
('Real Madrid'),
('Bayern');
''')
connection.commit() # Commit changes

## Visualise the tables

- Let's see our tables using `pandas`
- `read_sql` works fine with the `sqlite3` connection object

In [ ]:
pd.read_sql('SELECT * FROM players', connection)

In [ ]:
pd.read_sql('SELECT * FROM teams', connection)

## Types of joins

[![](figures/joins.webp){width="80%"}](#){data-modal-type="image" data-modal-url="figures/joins.webp"}

## Inner join

- **`INNER JOIN`** returns only the records that **match in both tables** (the intersection); rows without a match are dropped
- The matching condition is specified in the **`ON`** clause (e.g., `ON table1.id = table2.id`)
- Below, Haaland (`player_id` 6) is not in the `teams` table (which has 5 rows), so he doesn't appear in the result

In [ ]:
pd.read_sql('''
SELECT players.player_name, teams.team_name, players.goals, players.victories
FROM players
INNER JOIN teams
ON players.player_id = teams.team_id;
''', connection)

## Left join

- **`LEFT JOIN`** returns all records from the **left table** plus the matched records from the right table
- Where there is no match, columns from the right side come back as `NULL` (`None` in pandas)
- Probably the **most common join in practice**, because it keeps every row from the table you're focused on
- Haaland is included here because he is in the `players` table (the left one)

In [ ]:
pd.read_sql('''
SELECT players.player_name, teams.team_name, players.goals
FROM players
LEFT JOIN teams
ON players.player_id = teams.team_id;
''', connection)

## Right join

- **`RIGHT JOIN`** is the **mirror** of `LEFT JOIN`: all records from the right table plus matched records from the left
- Columns from the left side come back as `NULL` where there is no match
- Less common in practice (you can usually rewrite it as a `LEFT JOIN` with the tables swapped)
- In our data, every `team_id` matches a `player_id`, so the `RIGHT JOIN` happens to look identical to the `INNER JOIN`

In [ ]:
pd.read_sql('''
SELECT players.player_name, teams.team_name, players.goals
FROM players
RIGHT JOIN teams
ON players.player_id = teams.team_id;
''', connection)

## Full outer join

- **`FULL OUTER JOIN`** returns all records when there is a match in **either** the left or the right table
- The least common join in practice: results are large, often less focused, and more expensive to compute
- Syntax: `SELECT columns FROM table1 FULL OUTER JOIN table2 ON table1.column = table2.column`

In [ ]:
pd.read_sql('''
SELECT players.player_name, teams.team_name, players.goals
FROM players
FULL OUTER JOIN teams
ON players.player_id = teams.team_id;
''', connection)

## Setup for the exercise

- First, let's create two new tables (`products`, `reviews`) and insert some data. SQLite uses `REAL` for decimal numbers

In [ ]:
# Create the tables and insert data
cursor.execute('DROP TABLE IF EXISTS reviews;') 
cursor.execute('DROP TABLE IF EXISTS products;')
cursor.execute('''
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL,
    price REAL 
);
''')

# Insert products
cursor.execute('''
INSERT INTO products (product_name, price) VALUES
    ('Coffee Maker', 99.99),
    ('Toaster', 29.99),
    ('Blender', 79.99),
    ('Microwave', 149.99),
    ('Air Fryer', 89.99);
''')

cursor.execute('''
CREATE TABLE reviews (
    review_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id INT,
    rating INT CHECK (rating BETWEEN 1 AND 5),
    comment TEXT,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
''')

# Insert reviews
cursor.execute('''
INSERT INTO reviews (product_id, rating, comment) VALUES
    (1, 5, 'Great coffee maker!'),
    (1, 4, 'Good but expensive'),
    (2, 3, 'Average toaster'),
    (3, 5, 'Best blender ever');
''')
connection.commit()
print("Tables 'products' and 'reviews' created and populated.")

## Try it yourself! 🧠

- Now merge the `products` and `reviews` tables using `INNER JOIN` and `LEFT JOIN`
- Explain the differences between the two results based on which products appear
-

In [ ]:
# Your code here

# Special joins 🌟

## Cross join

- **`CROSS JOIN`** is available in SQL, including SQLite
- A cross join does not use any comparisons (like `ON`) to match rows
- Instead, the result pairs *every* row from the first table with *every* row from the second table (Cartesian product)
- Useful for generating all possible combinations (e.g., pairing all sizes with all colors)

In [ ]:
# Displaying cross join between players and teams
pd.read_sql('''
SELECT players.player_name, teams.team_name
FROM players
CROSS JOIN teams
ORDER BY players.player_id, teams.team_id;
''', connection)

## Cross join

- Here's another example generating T-shirt combinations. SQLite uses **`||`** for string concatenation, not `CONCAT()`

In [ ]:
# Drop and recreate tables
cursor.execute('DROP TABLE IF EXISTS colours;')
cursor.execute('DROP TABLE IF EXISTS sizes;')
cursor.execute('CREATE TABLE colours (colour_name TEXT);')
cursor.execute('CREATE TABLE sizes (size_code TEXT);')
cursor.execute("INSERT INTO colours VALUES ('Black'), ('Red');")
cursor.execute("INSERT INTO sizes VALUES ('S'), ('M');")
connection.commit()

# Perform cross join and concatenate strings using ||
pd.read_sql('''
SELECT
    colours.colour_name,
    sizes.size_code,
    colours.colour_name || ' - ' || sizes.size_code as t_shirt
FROM colours
CROSS JOIN sizes
ORDER BY colours.colour_name, sizes.size_code DESC;
''', connection)

## Self join

- A **self join** is a regular join, but the table is joined **with itself** 🤯
- Useful when a table contains relationships between rows *of the same type*. For example, an `employees` table where each row has a `manager_id` pointing to another `employee_id`
- Since the same table is referenced twice, **table aliases are required** to tell the two instances apart
- Pattern: `SELECT ... FROM employees AS e JOIN employees AS m ON e.manager_id = m.employee_id`. Here `e` is the employee and `m` is the manager

## Self join

- Let's see an example with a `family` table where `mother_id` refers back to `person_id`

In [ ]:
cursor.execute('DROP TABLE IF EXISTS family;')
cursor.execute('''
CREATE TABLE family (
    person_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    mother_id INT 
);
''')

cursor.execute('''
INSERT INTO family (name, mother_id) VALUES
    ('Emma', NULL), -- grandmother (id 1)
    ('Sarah', 1),   -- Emma's daughter (id 2)
    ('Lisa', 1),    -- Emma's daughter (id 3)
    ('Tom', 2),     -- Sarah's son (id 4)
    ('Alice', 2);   -- Sarah's daughter (id 5)
''')
connection.commit()

# Self join to find child-mother pairs
pd.read_sql('''
SELECT
    children.name as child,
    mothers.name as mother
FROM family AS children
JOIN family AS mothers ON children.mother_id = mothers.person_id
ORDER BY mothers.name, children.name;
''', connection)

## Self join

- Another example using the `players` table
- Goal: compare the `goals` of every player against every other player
- The condition **`p1.player_id < p2.player_id`** does two things:
  - Avoids duplicate pairs (so we don't get both Messi vs Vini Jr. and Vini Jr. vs Messi)
  - Skips comparing a player to themselves

In [ ]:
pd.read_sql('''
SELECT
    p1.player_name,
    p1.goals,
    p2.player_name as compared_to,
    p2.goals as their_goals,
    p1.goals - p2.goals as difference
FROM players AS p1
JOIN players AS p2
ON p1.player_id < p2.player_id -- Avoid duplicates and self-comparison
ORDER BY difference DESC;
''', connection)

## Try it yourself! 🧠

Using the `family` table (still loaded from the earlier example):

- Find all **sibling pairs**: people who share the same `mother_id`
- Use a **self join** on `family`, and apply the `f1.person_id < f2.person_id` trick to avoid duplicate pairs and self-matches
- Return each pair as two columns: `sibling_a` and `sibling_b`

-

In [ ]:
# Your code here

# Merge tables by row 🧩

## Union

- The **`UNION`** operator combines the result sets of two or more `SELECT` statements vertically (stacking rows)
- It automatically **removes duplicate rows** from the combined results. To keep duplicates, use `UNION ALL`
- Each `SELECT` must return the same number of columns, with compatible data types in corresponding positions
- Below we find players who scored more than 7 goals OR achieved more than 3 victories

In [ ]:
# Select players with more than 5 goals OR more than 3 victories
pd.read_sql('''
SELECT player_name, goals, victories
FROM players
WHERE goals > 7 

UNION -- Combines results and removes duplicates

SELECT player_name, goals, victories
FROM players
WHERE victories > 3

ORDER BY player_name;
''', connection)

## Union all and intersect

- **`UNION ALL`** also stacks two result sets vertically
- Unlike `UNION`, it **retains all duplicate rows**. It is generally faster than `UNION` because it does not check for duplicates

In [ ]:
# Combine players with > 7 goals and players with > 3 victories
# UNION ALL keeps all rows, including duplicates if a player meets both criteria.
pd.read_sql('''
SELECT player_name, goals, victories, 'High Scorer (>7)' AS category
FROM players
WHERE goals > 7

UNION ALL

SELECT player_name, goals, victories, 'Many Victories (>3)' AS category
FROM players
WHERE victories > 3

ORDER BY player_name, category;
''', connection)

## Intersect

- **`INTERSECT`** returns only the rows that are **common** to the result sets of both `SELECT` statements
- Like `UNION`, it removes duplicates within the final result
- Below we find players who are both high scorers (more than 7 goals) and have achieved many victories (more than 3)

In [ ]:
# Find players who are BOTH high scorers (goals > 7) AND have many victories (victories > 3)
pd.read_sql('''
SELECT player_name
FROM players
WHERE goals > 7

INTERSECT

SELECT player_name
FROM players
WHERE victories > 3

ORDER BY player_name;
''', connection)

## Except

- **`EXCEPT`** returns the rows from the first `SELECT` that are **not present** in the second. It finds the difference between the two sets
- Like `UNION`, it removes duplicates before returning the final result

In [ ]:
# Example for EXCEPT: Find players who scored more than 5 goals
# but did NOT achieve more than 3 victories.
pd.read_sql('''
SELECT player_name
FROM players
WHERE goals > 5

EXCEPT

SELECT player_name
FROM players
WHERE victories > 3

ORDER BY player_name;
''', connection)

## Try it yourself! 🧠

Using the `players` table:

1. Use **`UNION`** to list every player who has either `goals > 5` OR `victories > 3`. How many distinct players do you get?
2. Use **`EXCEPT`** to list players with `goals > 5` who did NOT achieve `victories > 3`. Who shows up?

-

In [ ]:
# Your code here

## UPSERT (`INSERT ... ON CONFLICT`)

- SQLite provides **`UPSERT`** (Update or Insert) operations using the `INSERT ... ON CONFLICT` clause
- This allows you to attempt an `INSERT`, and if it violates a constraint (like `UNIQUE` or `PRIMARY KEY`), you can specify an alternative action, typically a `DO UPDATE`
- You can specify different actions for different types of conflicts, such as `DO NOTHING`, `REPLACE`, or `DO UPDATE SET`
- Let's see a simplified example: If we try to insert a player with an existing `player_name`, we update their `goals` instead
- You will notice that `excluded` is a special table that refers to the row that would have been inserted if there was no conflict. **SQLite always uses this name**

In [ ]:
player_data = ('Messi', 2, 1)

sql_upsert = """ 
INSERT INTO players (player_name, goals, victories) VALUES (?, ?, ?) 
ON CONFLICT(player_name) DO UPDATE SET 
goals = goals + excluded.goals, 
victories = victories + excluded.victories;
"""
cursor.execute(sql_upsert, player_data)
pd.read_sql('SELECT * FROM players', connection)

# Views 🔎

## Views

- A **`VIEW`** is a **virtual table** built from a `SELECT` query
- It does not store data itself; it simply runs the query whenever you read from it
- Useful for **simplifying complex queries** or saving the ones you reuse often
- Create one with **`CREATE VIEW`**, remove it with **`DROP VIEW`**

In [ ]:
# Drop the view if it exists
cursor.execute('DROP VIEW IF EXISTS player_stats;')

# Create the view
cursor.execute('''
CREATE VIEW player_stats AS
SELECT player_name, SUM(goals) AS total_goals, SUM(victories) AS total_victories
FROM players
GROUP BY player_name;
''')
connection.commit()

pd.read_sql('SELECT * FROM player_stats LIMIT 4', connection)

## Views

- You can also use the view in a join with another table. For example, let's create a view that combines colours and sizes of T-shirts as we did before

In [ ]:
# Drop the view if it already exists to avoid errors on re-run
cursor.execute('DROP VIEW IF EXISTS colour_size;')

# Create the view using cursor.execute()
cursor.execute('''
CREATE VIEW colour_size AS
SELECT
    c.colour_name,
    s.size_code,
    c.colour_name || ' - ' || s.size_code as t_shirt -- Use || for concatenation
FROM colours AS c
CROSS JOIN sizes AS s
ORDER BY c.colour_name, s.size_code DESC;
''')
connection.commit() # Commit the view creation

# Now showcase the view by querying it with pandas
pd.read_sql('SELECT * FROM colour_size;', connection)

# Conclusion 📖

## Conclusion

- We covered the four core joins (`INNER`, `LEFT`, `RIGHT`, `FULL OUTER`) and two special ones (`CROSS JOIN`, `SELF JOIN`)
- We merged tables vertically with `UNION`, `UNION ALL`, `INTERSECT`, and `EXCEPT`
- We saw SQLite's UPSERT clause (`INSERT ... ON CONFLICT`) and how to create reusable `VIEW`s
- That's a lot for one lecture! 🚀 The exercises in the appendix will help you practise

# And that's all for today! 🎉

# Thank you and have a great rest of your day! 🙏

## Appendix 01

- Here is the solution to the exercise comparing `INNER JOIN` and `LEFT JOIN` for `products` and `reviews`

In [ ]:
# INNER JOIN only includes products that have at least one review.
# Products like 'Microwave' and 'Air Fryer' are excluded.
pd.read_sql('''
    SELECT p.product_name, r.rating, r.comment
    FROM products p
    INNER JOIN reviews r ON p.product_id = r.product_id
    ORDER BY p.product_id, r.review_id;
''', connection)

In [ ]:
# LEFT JOIN keeps every product, with NULL ratings/comments where there is no review.
pd.read_sql('''
    SELECT p.product_name, r.rating, r.comment
    FROM products p
    LEFT JOIN reviews r ON p.product_id = r.product_id
    ORDER BY p.product_id, r.review_id;
''', connection)

## Appendix 02

Part 1, `UNION` (distinct players satisfying either condition):

In [ ]:
pd.read_sql('''
SELECT player_name FROM players WHERE goals > 5
UNION
SELECT player_name FROM players WHERE victories > 3
ORDER BY player_name;
''', connection)

Three distinct players: Messi, Neymar, Vini Jr

Part 2, `EXCEPT` (goals > 5 but NOT victories > 3):

In [ ]:
pd.read_sql('''
SELECT player_name FROM players WHERE goals > 5
EXCEPT
SELECT player_name FROM players WHERE victories > 3
ORDER BY player_name;
''', connection)

Just Neymar: he scored more than 5 goals but did not pass the 3-victories threshold

## Appendix 03

Self join on `family`, using two aliases (`f1`, `f2`) and the `<` trick to skip duplicates and self-matches:

In [ ]:
pd.read_sql('''
SELECT
    f1.name AS sibling_a,
    f2.name AS sibling_b
FROM family AS f1
JOIN family AS f2
    ON f1.mother_id = f2.mother_id
    AND f1.person_id < f2.person_id
ORDER BY sibling_a;
''', connection)

Two pairs: Sarah-Lisa (both children of Emma) and Tom-Alice (both children of Sarah)